In [2]:
import pandas as pd
import numpy as np

postings = pd.read_csv('/Users/zacharysoriano/Personal/postings/postings.csv')

In [3]:
# cleaning the data

postings_clean = postings.drop(columns = ['fips', 'zip_code', 'normalized_salary', 'sponsored', 'posting_domain', 'listed_time', 'closed_time',
                                          'expiry', 'application_type', 'application_url', 'job_posting_url', 'original_listed_time', 'job_id',
                                           'remote_allowed', 'currency', 'views', 'company_id', 'applies', 'med_salary', 'skills_desc', 'work_type',
                                          'compensation_type'])
postings_clean = postings_clean.dropna()
postings_clean.reset_index(drop=True, inplace=True)
postings_clean.head(15)

,company_name,title,description,max_salary,pay_period,location,min_salary,formatted_work_type,formatted_experience_level
0,"I.T. Solutions, Inc.","Validation Engineer, Labware LIMS","Validation Engineer, Labware LIMSFoster City, ...",70.0,HOURLY,"Foster City, CA",60.0,Contract,Mid-Senior level
1,ActOne Group,Administrative Assistant - CONCUR,Global Financial Services firm is seeking an e...,90000.0,YEARLY,"New York, NY",75000.0,Full-time,Associate
2,ABC Farigua Division,Customer Service Representative,We are seeking future agents to join our team!...,105000.0,YEARLY,"Greater Orlando, FL",75000.0,Full-time,Entry level
3,TECHEAD,Inbound Call Center Specialist,"Always Connecting, Always Evolving.\nIf you ar...",19.0,HOURLY,"Richmond, VA",18.0,Contract,Associate
4,Prolink,Tool and Die Maker,Job Summary:The Tool and Die Maker will build ...,35.0,HOURLY,Cincinnati Metropolitan Area,32.0,Full-time,Associate
5,"Dexterity, Inc.",Senior Mechanical Engineer,Senior Mechanical Engineer\nLocation: Redwood ...,190000.0,YEARLY,"Redwood City, CA",150000.0,Full-time,Mid-Senior level
6,Staff Management | SMX,Quality Engineer,Staff Management | SMX is currently in search ...,43.0,HOURLY,"New Albany, IN",28.0,Full-time,Associate
7,Ascendion,Quality Assurance Specialist,About Ascendion Ascendion is a full-service di...,72000.0,YEARLY,"Seattle, WA",60000.0,Full-time,Associate
8,Premier Brands Group Holdings,"Technical Designer, Womans Denim Bottoms","PREMIER BRANDS GROUP HOLDINGSIconic by Nature,...",95000.0,YEARLY,New York City Metropolitan Area,75000.0,Full-time,Mid-Senior level
9,Insight Global,Flight Software Engineer,Must Haves:10-15 years of experience with C++ ...,170000.0,YEARLY,"Webster, TX",140000.0,Full-time,Mid-Senior level


In [4]:
# make all salaries yearly
postings_clean['pay_period'].unique()
for i in range(len(postings_clean)):
    if postings_clean.loc[i, 'pay_period'] == 'YEARLY':
        postings_clean.loc[i, 'norm_max_salary'] = postings_clean.loc[i, 'max_salary'] 
        postings_clean.loc[i, 'norm_min_salary'] = postings_clean.loc[i, 'min_salary']
    elif postings_clean.loc[i, 'pay_period'] == 'HOURLY':
        postings_clean.loc[i, 'norm_max_salary'] = postings_clean.loc[i, 'max_salary'] * 40 * 52
        postings_clean.loc[i, 'norm_min_salary'] = postings_clean.loc[i, 'min_salary'] * 40 * 52
    elif postings_clean.loc[i, 'pay_period'] == 'MONTHLY':
        postings_clean.loc[i, 'norm_max_salary'] = postings_clean.loc[i, 'max_salary'] * 12
        postings_clean.loc[i, 'norm_min_salary'] = postings_clean.loc[i, 'min_salary'] * 12
    elif postings_clean.loc[i, 'pay_period'] == 'WEEKLY':
        postings_clean.loc[i, 'norm_max_salary'] = postings_clean.loc[i, 'max_salary'] * 52
        postings_clean.loc[i, 'norm_min_salary'] = postings_clean.loc[i, 'min_salary'] * 52
    # biweekly
    else:
        postings_clean.loc[i, 'norm_max_salary'] = postings_clean.loc[i, 'max_salary'] * 26
        postings_clean.loc[i, 'norm_min_salary'] = postings_clean.loc[i, 'min_salary'] * 26
postings_clean['salary_midpoint'] = (postings_clean['norm_max_salary'] + postings_clean['norm_min_salary']) / 2
len(postings_clean)
# to predict



23002

In [5]:
# log transform salary
postings_clean['log_salary_midpoint'] = np.log(postings_clean['salary_midpoint'])
y = postings_clean['log_salary_midpoint']


In [6]:
# let's only consider jobs with salaries from 10,000 to 500,000 dollars
salary = postings_clean['salary_midpoint']
postings_clean = postings_clean[(salary > 10000) & (salary < 500000)]
len(postings_clean)

# eliminated ~ 400 observations

22636

In [7]:
postings_clean = postings_clean.drop(columns = ['min_salary', 'pay_period', 'max_salary', 'formatted_work_type', 'formatted_experience_level',
                     'norm_min_salary', 'norm_max_salary'])

In [8]:
# make float32 (standard)
postings_clean['log_salary_midpoint'] = postings_clean['log_salary_midpoint'].astype('float32')

In [9]:
postings_clean

,company_name,title,description,location,salary_midpoint,log_salary_midpoint
0,"I.T. Solutions, Inc.","Validation Engineer, Labware LIMS","Validation Engineer, Labware LIMSFoster City, ...","Foster City, CA",135200.0,11.814510
1,ActOne Group,Administrative Assistant - CONCUR,Global Financial Services firm is seeking an e...,"New York, NY",82500.0,11.320554
2,ABC Farigua Division,Customer Service Representative,We are seeking future agents to join our team!...,"Greater Orlando, FL",90000.0,11.407565
3,TECHEAD,Inbound Call Center Specialist,"Always Connecting, Always Evolving.\nIf you ar...","Richmond, VA",38480.0,10.557894
4,Prolink,Tool and Die Maker,Job Summary:The Tool and Die Maker will build ...,Cincinnati Metropolitan Area,69680.0,11.151669
...,...,...,...,...,...,...
22997,"TalentBurst, an Inc 5000 company",Contract Administrator,"Position: Clinical Contracts Analyst, Req#: 63...","Irvine, CA",83200.0,11.329002
22998,Athena Recruiting,Catering Event Manager,This role handles all the onsite catering and ...,Greater Indianapolis,57500.0,10.959540
22999,"TalentBurst, an Inc 5000 company",Quality Engineer,Position: Quality Engineer I (Complaint Invest...,"Irvine, CA",83200.0,11.329002
23000,Lozano Smith,Title IX/Investigations Attorney,Our Walnut Creek office is currently seeking a...,"Walnut Creek, CA",157500.0,11.967181


In [15]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(postings_clean, test_size=0.2, random_state=1)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=1)


In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

text_col = 'description' 
target_col = 'log_salary_midpoint'

X_train_raw = train_df[text_col]
y_train = train_df[target_col]
X_val_raw = val_df[text_col]
y_val = val_df[target_col]

bow = CountVectorizer(max_features=5000, stop_words='english')

X_train_bow = bow.fit_transform(X_train_raw)
X_val_bow = bow.transform(X_val_raw)

baseline_model_bow = LinearRegression()
baseline_model_bow.fit(X_train_bow, y_train)

# 4. Evaluate
y_pred_bow = baseline_model_bow.predict(X_val_bow)

mae_bow = mean_absolute_error(y_val, y_pred_bow)
r2_bow = r2_score(y_val, y_pred_bow)

print(f"BoW Baseline MAE: {mae_bow:.2f}")
print(f"BoW Baseline R^2 Score: {r2_bow:.4f}")

BoW Baseline MAE: 0.27
BoW Baseline R^2 Score: 0.5173


In [10]:
from datasets import Dataset

raw_train_ds = Dataset.from_pandas(train_df)
raw_val_ds = Dataset.from_pandas(val_df)
raw_test_ds = Dataset.from_pandas(test_df)

ds = {"train": raw_train_ds, "validation": raw_val_ds, "test": raw_test_ds}

In [11]:
import os
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

transformers.utils.logging.disable_progress_bar()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

BASE_MODEL = "distilbert-base-uncased"
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 4

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)



/opt/anaconda3/envs/salary-bert/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["description"], 
        truncation=True, 
        padding="max_length", 
        max_length=256
    )
    
    tokenized_inputs["labels"] = [float(x) for x in examples['log_salary_midpoint']]
    
    return tokenized_inputs

for split in ds:
    ds[split] = ds[split].map(
        preprocess_function, 
        batched=True,          
        remove_columns=ds[split].column_names 
    )

Map:   0%|          | 0/18108 [00:00<?, ? examples/s]

Map:   0%|          | 0/2264 [00:00<?, ? examples/s]

Map:   0%|          | 0/2264 [00:00<?, ? examples/s]

In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

def compute_metrics_for_regression(eval_pred):
    logits, labels = eval_pred
    labels = labels.reshape(-1, 1)
    
    mse = mean_squared_error(labels, logits)
    mae = mean_absolute_error(labels, logits)
    r2 = r2_score(labels, logits)
    single_squared_errors = ((logits - labels).flatten()**2).tolist()
    
    
    accuracy = sum([1 for e in single_squared_errors if e < 0.25]) / len(single_squared_errors)
    
    return {"mse": mse, "mae": mae, "r2": r2, "accuracy": accuracy}

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./models/distilbert-salary-regression",
    
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    
   
    eval_strategy="epoch", 
    save_strategy="epoch",
  
    use_mps_device=True, 
    
    save_total_limit=2,
    metric_for_best_model="accuracy", 
    load_best_model_at_end=True,
    weight_decay=0.01,
    
    logging_steps=100, 
)

/opt/anaconda3/envs/salary-bert/lib/python3.10/site-packages/transformers/training_args.py:2199: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(


In [15]:
import torch

class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs[0][:, 0]
        loss = torch.nn.functional.mse_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [16]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = RegressionTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics_for_regression,
    data_collator=data_collator,
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mse,Mae,R2,Accuracy
1,0.239400,0.112744,0.112744,0.259371,0.579735,0.867491
2,0.193400,0.108602,0.108602,0.259189,0.595174,0.882067
3,0.162000,0.083951,0.083951,0.218158,0.687064,0.915636
4,0.154300,0.096688,0.096688,0.239834,0.639584,0.902827


TrainOutput(global_step=4528, training_loss=1.3030646960432033, metrics={'train_runtime': 4694.3463, 'train_samples_per_second': 15.43, 'train_steps_per_second': 0.965, 'total_flos': 4797353754206208.0, 'train_loss': 1.3030646960432033, 'epoch': 4.0})

In [18]:
import os
save_path = "./salary_bert_final"
if not os.path.exists(save_path):
    os.makedirs(save_path)


trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model successfully saved to {save_path}")

Model successfully saved to ./salary_bert_final


In [26]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np

path = "./salary_bert_final"
model = AutoModelForSequenceClassification.from_pretrained(path).to("mps")
tokenizer = AutoTokenizer.from_pretrained(path)

def predict_salary(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to("mps")
    with torch.no_grad():
        log_pred = model(**inputs).logits.item()
    return np.exp(log_pred)


In [28]:
# test
job_text = ""
print(f"Predicted Salary: ${predict_salary(job_text):,.2f}")

Predicted Salary: $132,711.61
